# 02 — Skill Extraction Experiments

**Purpose:** Prototype and evaluate skill extraction approaches.  
Start with keyword matching, then explore TF-IDF-based extraction.

---

In [ ]:
import json
import re
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path
from collections import Counter

print('Libraries loaded.')

In [ ]:
# Load data
DATA_PATH = Path('..') / 'data' / 'sample' / 'sample_jobs.csv'
SKILLS_PATH = Path('..') / 'data' / 'sample' / 'skills_dictionary.json'

df = pd.read_csv(DATA_PATH)
with open(SKILLS_PATH, 'r') as f:
    skills_dict = json.load(f)

print(f'Jobs loaded: {len(df)}')
print(f'Skill categories: {list(skills_dict.keys())}')

In [ ]:
# Helper: clean text
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['clean_description'] = df['description'].apply(clean_text)
df[['job_title', 'clean_description']].head(3)

In [ ]:
# Keyword-based skill extraction
def extract_skills(text, skills_dict):
    found = {}
    for category, skills in skills_dict.items():
        matched = [skill for skill in skills if skill in text]
        if matched:
            found[category] = matched
    return found

# Test on one job
sample_text = df.loc[0, 'clean_description']
print(f'Job: {df.loc[0, "job_title"]}')
print('Extracted skills:')
print(extract_skills(sample_text, skills_dict))

In [ ]:
# Apply to all jobs
df['extracted_skills'] = df['clean_description'].apply(
    lambda text: extract_skills(text, skills_dict)
)

# Flatten all found skills across jobs
all_skills = []
for skills_found in df['extracted_skills']:
    for category, skills in skills_found.items():
        all_skills.extend(skills)

skill_counts = Counter(all_skills)
print(f'Total unique skills found: {len(skill_counts)}')
print('\nTop 15 skills:')
for skill, count in skill_counts.most_common(15):
    print(f'  {skill}: {count}')

In [ ]:
# Visualise top skills
top_skills_df = pd.DataFrame(skill_counts.most_common(20), columns=['skill', 'count'])

fig = px.bar(
    top_skills_df,
    x='count',
    y='skill',
    orientation='h',
    title='Top 20 Skills in Job Postings (Keyword Matching)',
    color='count',
    color_continuous_scale='Viridis'
)
fig.update_layout(yaxis=dict(autorange='reversed'))
fig.show()

In [ ]:
# TF-IDF approach (prototype)
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=50, stop_words='english', ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(df['clean_description'])

feature_names = vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names, index=df['job_title'])

print('TF-IDF matrix shape:', tfidf_df.shape)
tfidf_df.head(3)

---
## Observations

- Keyword matching works well for known skill terms from the dictionary
- TF-IDF captures important unigrams and bigrams beyond the fixed vocabulary
- Next step: combine both approaches and implement in `src/skill_extraction.py`

## Next Steps
- Experiment with `sentence-transformers` for semantic matching
- Build job clustering prototype in the next notebook